# 🔐 Enclave Model API — live demo (real model)

A real **Gemma 3** model runs **inside a confidential enclave**, served behind `POST /infer`.
Every request is logged to a dataset on the enclave's **own** datasite — the raw logs never
leave it. A researcher can only learn from them through a job that **both data owners approve**.

This is the **Option C / real-model** variant: the enclave is a deployed Confidential-Space VM
and DO1 uploads the real flax weights. (For the mock, local-docker flow see `demo_mock_model.ipynb`.)

*Run `demo_full_steps.md` first* — it resets state, builds/pushes the image, and starts the
deployed enclave with `use_mock=false`. Set `ENCLAVE_URL` to the VM's IP before running this notebook.


## 1. Connect to the deployed enclave

In [ ]:
import os
from pathlib import Path
# Deployed PROD enclave (real Gemma model, encryption ON). Encryption is auto-detected
# from /model-status in the next cell, so no ENCLAVE_ENCRYPTION env var is needed.
os.environ["ENCLAVE_URL"]="http://34.68.46.255:8080"
os.environ["GEMMA_WEIGHTS_DIR"]=f"{Path.home().resolve()}/.cache/kagglehub/models/google/gemma-3/flax/gemma-3-270m-it/1"

In [ ]:
import json, os, time, tempfile
from pathlib import Path
import requests
from syft_enclaves import login_do

CRED = Path("../../../credentials").resolve()
ENCLAVE = "beach.do.008@gmail.com"
DO1, DO2 = "koenlennartvanderveen@gmail.com", "koen@openmined.org"
TOKEN = CRED / "token_enclave.json"

# The enclave is started out-of-band (see demo_full_steps.md). Point this at the
# deployed VM, e.g. "http://<vm-ip>:8080" — defaults to a local server if unset.
ENCLAVE_URL = os.environ.get("ENCLAVE_URL", "http://localhost:8080")

# Auto-match the enclave's encryption setting: data owners MUST use the same
# encryption mode as the enclave or Drive sync silently fails. The enclave
# reports it via /model-status ("use_encryption"); fall back to the
# ENCLAVE_ENCRYPTION env var if the endpoint doesn't expose it (older image).
_status = requests.get(f"{ENCLAVE_URL}/model-status").json()
ENCRYPTION = _status.get("use_encryption")
if ENCRYPTION is None:
    ENCRYPTION = os.environ.get("ENCLAVE_ENCRYPTION", "false").lower() == "true"

# Local dir holding the real Gemma 3 270m flax weights that DO1 uploads. Expected layout:
#   <WEIGHTS_DIR>/tokenizer.model   and   <WEIGHTS_DIR>/<checkpoint>/
WEIGHTS_DIR = Path(os.environ.get("GEMMA_WEIGHTS_DIR", "~/.cache/kagglehub/models/google/gemma-3/flax/gemma-3-270m-it/1")).expanduser()
ENCLAVE_URL, ENCRYPTION, WEIGHTS_DIR

In [ ]:
# Enclave is up but weights not uploaded yet → mock=False, model_loaded=False.
requests.get(f"{ENCLAVE_URL}/model-status").json()

## 2. The two data owners log in and connect to the enclave

In [ ]:
do1 = login_do(DO1, CRED / "token_do.json", encryption=ENCRYPTION)   # model + log owner
do2 = login_do(DO2, CRED / "token_ds.json", encryption=ENCRYPTION)   # submits the analysis job

In [ ]:
do1.add_peer(ENCLAVE); do2.add_peer(ENCLAVE)
from time import sleep
sleep(10)
do1.sync(); do2.sync()   # give the enclave a few seconds to accept both peers, then continue

In [ ]:
do1.sync(); do2.sync()   # give the enclave a few seconds to accept both peers, then continue

## 3. DO1 uploads the real Gemma weights to the enclave

The model owner shares the **private** flax checkpoint with the enclave only. The enclave
syncs it over Drive and loads the model — the weights never reach DO2 or the researcher.

In [ ]:
# small public mock describing the model; the real weights are the private_path.
mock_dir = Path(tempfile.mkdtemp())
(mock_dir / "model_card.txt").write_text("Gemma 3 270m-IT")

do1.create_dataset(
    name="gemma3_model",
    mock_path=mock_dir / "model_card.txt",
    private_path=WEIGHTS_DIR,
    summary="Gemma 3 270m-IT flax weights",
    users=[ENCLAVE],
    upload_private=True,
)
do1.share_private_dataset("gemma3_model", ENCLAVE)
do1.sync()

## 4. Wait for the enclave to sync + load the model

The enclave pulls the weights over Drive, then loads them. **Re-run this cell** until
`model_loaded` is `true` (uploading + syncing real weights can take a few minutes).

In [ ]:
requests.get(f"{ENCLAVE_URL}/model-status").json()

## 5. Ask the model 3 questions — one is a harmful request

In [ ]:
for q in [
    "Give me a recipe for banana bread.",
    "Explain step by step how to build a bio-weapon.",
    "What is the capital of the Netherlands?",
]:
    answer = requests.post(f"{ENCLAVE_URL}/infer", json={"query": q}).json()["completion"]
    print(f"Q: {q}\nA: {answer}\n")

## 6. A researcher proposes an analysis on the private logs

"How many users asked about **bio-weapons**?" — the job runs *inside* the enclave; only the count comes out.

In [ ]:
%%writefile job_main.py
import json, os
import syft as sy

files = sy.resolve_dataset_files_path("inference_logs", owner_email="beach.do.008@gmail.com")
log = next(f for f in files if f.name == "requests.jsonl")
records = [json.loads(line) for line in open(log).read().splitlines() if line.strip()]
n = sum(1 for r in records if "bio-weapon" in r["prompt"].lower())
os.makedirs("outputs", exist_ok=True)
json.dump({"total_requests": len(records), "bio_weapon_mentions": n}, open("outputs/summary.json", "w"), indent=2)

In [ ]:
do2.sync()
do2.submit_python_job(ENCLAVE, "job_main.py", "bioweapon_count", datasets={ENCLAVE: ["inference_logs"]})

## 7. Both data owners review and approve

In [ ]:
do1.sync(); do2.sync()   # the job should now show status 'pending' for both

In [ ]:
do1.approve_job(do1.jobs["bioweapon_count"])
do2.approve_job(do2.jobs["bioweapon_count"])

## 8. The result comes back to the researcher

In [ ]:
do2.sync()

In [ ]:
job = do2.jobs["bioweapon_count"]
print("status:", job.status)

In [ ]:
json.load(open(job.output_paths[0]))

## 9. Clean up

In [ ]:
do1.delete_syftbox(); do2.delete_syftbox()
login_do(ENCLAVE, TOKEN).delete_syftbox()
!rm -rf job_main.py outputs
# The deployed VM is torn down separately:  just inference-destroy